<a href="https://colab.research.google.com/github/Padmadharshini03/Machine-learning-/blob/main/experiment_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Program 3: Decision tree using the ID3 algorithm (PlayTennis data set)
import math
from collections import Counter
import pandas as pd

# ---------------- Training data ----------------
data = pd.DataFrame({
    'Outlook':     ['Sunny', 'Sunny', 'Overcast', 'Rain', 'Rain', 'Rain', 'Overcast',
                    'Sunny', 'Sunny', 'Rain', 'Sunny', 'Overcast', 'Overcast', 'Rain'],
    'Temperature': ['Hot', 'Hot', 'Hot', 'Mild', 'Cool', 'Cool', 'Cool',
                    'Mild', 'Cool', 'Mild', 'Mild', 'Mild', 'Hot', 'Mild'],
    'Humidity':    ['High', 'High', 'High', 'High', 'Normal', 'Normal', 'Normal',
                    'High', 'Normal', 'Normal', 'Normal', 'High', 'Normal', 'High'],
    'Wind':        ['Weak', 'Strong', 'Weak', 'Weak', 'Weak', 'Strong', 'Strong',
                    'Weak', 'Weak', 'Weak', 'Strong', 'Strong', 'Weak', 'Strong'],
    'PlayTennis':  ['No', 'No', 'Yes', 'Yes', 'Yes', 'No', 'Yes',
                    'No', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'No']
})
print("Training data:")
print(data)
print()


# ---------------- Entropy and Information Gain ----------------
def entropy(labels):
    counts = Counter(labels)
    total = len(labels)
    return -sum((c / total) * math.log2(c / total) for c in counts.values())


def information_gain(df, attribute, target):
    total_entropy = entropy(df[target])
    weighted = 0.0
    for _, subset in df.groupby(attribute):
        weighted += (len(subset) / len(df)) * entropy(subset[target])
    return total_entropy - weighted


# ---------------- ID3 algorithm ----------------
def id3(df, target, attributes):
    labels = df[target]
    if labels.nunique() == 1:              # all examples have the same class
        return labels.iloc[0]
    if not attributes:                     # no attributes left -> majority class
        return labels.mode()[0]

    gains = {a: information_gain(df, a, target) for a in attributes}
    best = max(gains, key=gains.get)       # attribute with highest gain
    tree = {best: {}}
    for value, subset in df.groupby(best):
        remaining = [a for a in attributes if a != best]
        tree[best][value] = id3(subset, target, remaining)
    return tree


def print_tree(tree, indent=''):
    if not isinstance(tree, dict):
        print(indent + '-> ' + str(tree))
        return
    attribute = next(iter(tree))
    for value, subtree in tree[attribute].items():
        print(f"{indent}{attribute} = {value}")
        print_tree(subtree, indent + '    ')


def classify(tree, sample):
    if not isinstance(tree, dict):
        return tree
    attribute = next(iter(tree))
    branch = tree[attribute].get(sample[attribute])
    if branch is None:
        return 'Unknown (value not seen in training)'
    return classify(branch, sample)


target = 'PlayTennis'
features = [c for c in data.columns if c != target]

print("Entropy of the whole data set: %.4f" % entropy(data[target]))
print("\nInformation gain of each attribute:")
for f in features:
    print(f"  {f:12s}: {information_gain(data, f, target):.4f}")

tree = id3(data, target, features)
print("\nDecision tree:")
print_tree(tree)
print("\nTree as dictionary:")
print(tree)

# ---------------- Classify a new sample ----------------
new_sample = {'Outlook': 'Sunny', 'Temperature': 'Cool', 'Humidity': 'High', 'Wind': 'Strong'}
print("\nNew sample:", new_sample)
print("Predicted class (PlayTennis):", classify(tree, new_sample))


Training data:
     Outlook Temperature Humidity    Wind PlayTennis
0      Sunny         Hot     High    Weak         No
1      Sunny         Hot     High  Strong         No
2   Overcast         Hot     High    Weak        Yes
3       Rain        Mild     High    Weak        Yes
4       Rain        Cool   Normal    Weak        Yes
5       Rain        Cool   Normal  Strong         No
6   Overcast        Cool   Normal  Strong        Yes
7      Sunny        Mild     High    Weak         No
8      Sunny        Cool   Normal    Weak        Yes
9       Rain        Mild   Normal    Weak        Yes
10     Sunny        Mild   Normal  Strong        Yes
11  Overcast        Mild     High  Strong        Yes
12  Overcast         Hot   Normal    Weak        Yes
13      Rain        Mild     High  Strong         No

Entropy of the whole data set: 0.9403

Information gain of each attribute:
  Outlook     : 0.2467
  Temperature : 0.0292
  Humidity    : 0.1518
  Wind        : 0.0481

Decision tree:
Outloo